# Injected Pulsar Recovery Check

This notebook reads `generated_pulsar_tests/metadata.json`, selects one generated case, refits the timing parameters with JUG starting from the sampler PAR, and compares the recovered values to the injected truth.

It is intended as a sampler-independent sanity check: if the injection and timing-model fit are consistent, `refit_minus_truth / fit_sigma` should usually be of order unity for parameters with enough leverage. Red noise/white noise realization scatter means exact zero is not expected.

In [ ]:
from pathlib import Path
import json
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from jug.engine.session import TimingSession

pd.set_option("display.precision", 6)

## Select Case

Set `PULSAR_OR_CASE` to an exact case name (`j1909_f0_f1`, `j1909_f0_f1_dm`) or a pulsar substring (`J1909-3744`, `B1953+29`, `J0030+0451`). For substring matches the notebook picks a simple spin case by default, because global DM is degenerate with DMX in these NG15 files.

In [ ]:
METADATA_PATH = Path("/home/mattm/soft/generated_pulsar_tests/metadata.json")
PULSAR_OR_CASE = "j1909_f0_f1"  # exact case name or pulsar substring
MAX_ITER = 12

metadata = json.loads(METADATA_PATH.read_text())
cases = {case["case"]: case for case in metadata["cases"]}
print("Available cases:")
for name in cases:
    print("  ", name)

def select_case(query):
    q = query.lower()
    if query in cases:
        return cases[query], [query]
    matches = []
    for name, record in cases.items():
        haystack = " ".join([
            name,
            record.get("source", ""),
            record.get("truth_par", ""),
            record.get("sampler_par", ""),
        ]).lower()
        if q in haystack:
            matches.append(name)
    if not matches:
        raise ValueError(f"No generated case matched {query!r}")
    preferred_order = ["f0_f1", "f1", "f0", "f0_f1_dm"]
    selected = matches[0]
    for token in preferred_order:
        for name in matches:
            if token in name:
                selected = name
                break
        if selected != matches[0] or token in selected:
            break
    return cases[selected], matches

case, matches = select_case(PULSAR_OR_CASE)
CASE_NAME = case["case"]
if len(matches) > 1:
    print(f"\nMultiple matches for {PULSAR_OR_CASE!r}; selecting {CASE_NAME!r}:")
    for name in matches:
        print("  ", name)

truth_par = Path(case["truth_par"])
sampler_par = Path(case["sampler_par"])
tim = Path(case["tim"])
fit_params = [p["param"] for p in case["perturbations"]]

print(f"\nSelected: {CASE_NAME}")
print("truth_par :", truth_par)
print("sampler_par:", sampler_par)
print("tim       :", tim)
print("fit_params:", fit_params)
print("noise:", json.dumps(case["noise"], indent=2))

## Prefit Check

Compute residuals under the truth PAR and sampler PAR before fitting. The truth PAR should be closer to the injected noisy data than the sampler PAR, modulo stochastic noise and covariances.

In [ ]:
truth_session = TimingSession(str(truth_par), str(tim), verbose=False)
sampler_session = TimingSession(str(sampler_par), str(tim), verbose=False)

truth_res = truth_session.compute_residuals(subtract_tzr=False)
sampler_res = sampler_session.compute_residuals(subtract_tzr=False)

print(f"Truth PAR residual RMS : {truth_res['rms_us']:.6g} us")
print(f"Sampler PAR residual RMS: {sampler_res['rms_us']:.6g} us")
print(f"Metadata truth RMS      : {case['truth_residual_rms_us']:.6g} us")

## JUG GLS Refit With Known Injected Red Noise

This is the main sampler-independent refit test. The generated sampler PARs intentionally do not contain red-noise lines, because the data files are meant for samplers that should infer noise. For this validation only, the notebook writes a temporary PAR with the known injected `TNRedAmp/TNRedGam/TNRedC` from metadata and runs the JUG GLS fitter.

Read `z_truth = (refit - truth) / fit_sigma`:

- near 0: refit recovered truth
- within about ±2: statistically consistent for this covariance
- large: either fit/model issue or parameter degeneracy

Note: global `DM` is not a clean standalone recovery test in these NG15 files because they contain many DMX bins and JUG auto-fits DMX. If you choose a DM case, the notebook will warn you.

In [ ]:
def write_gls_fit_par_with_known_red_noise(case, sampler_par):
    red = case["noise"]["red"]
    text = Path(sampler_par).read_text()
    text += (
        f"\n# Added by injected_recovery_check.ipynb for GLS validation only\n"
        f"TNRedAmp {red['log10_A']}\n"
        f"TNRedGam {red['gamma']}\n"
        f"TNRedC {red['n_harmonics']}\n"
    )
    tmp = Path(tempfile.gettempdir()) / f"{case['case']}_gls_known_red.par"
    tmp.write_text(text)
    return tmp

def has_dmx_ranges(par_path):
    for line in Path(par_path).read_text().splitlines():
        parts = line.split()
        if parts and (parts[0].startswith("DMXR1_") or parts[0].startswith("DMX_")):
            return True
    return False

if "DM" in fit_params and has_dmx_ranges(sampler_par):
    print("WARNING: This case fits global DM but the PAR also has DMX ranges.")
    print("JUG auto-fits DMX bins, so global DM can be degenerate and z_truth may be misleading.")

gls_par = write_gls_fit_par_with_known_red_noise(case, sampler_par)
print("Temporary GLS PAR:", gls_par)

gls_session = TimingSession(str(gls_par), str(tim), verbose=False)
gls_fit_result = gls_session.fit_parameters(
    fit_params=fit_params,
    max_iter=MAX_ITER,
    verbose=False,
)

print("converged:", gls_fit_result.get("converged"))
print("iterations:", gls_fit_result.get("iterations"))
print(f"prefit RMS : {gls_fit_result.get('prefit_rms', np.nan):.6g} us")
print(f"postfit RMS: {gls_fit_result.get('final_rms', np.nan):.6g} us")

gls_rows = []
for pinfo in case["perturbations"]:
    key = pinfo["param"]
    sampler_value = float(pinfo["sampler_value"])
    truth_value = float(pinfo["true_value"])
    refit_value = float(gls_fit_result["final_params"].get(key, np.nan))
    fit_sigma = float(gls_fit_result.get("uncertainties", {}).get(key, np.nan))
    refit_minus_truth = refit_value - truth_value
    z_truth = refit_minus_truth / fit_sigma if np.isfinite(fit_sigma) and fit_sigma > 0 else np.nan
    gls_rows.append({
        "param": key,
        "sampler_value": sampler_value,
        "truth_value": truth_value,
        "gls_refit_value": refit_value,
        "expected_truth_minus_sampler": truth_value - sampler_value,
        "gls_refit_minus_sampler": refit_value - sampler_value,
        "gls_refit_minus_truth": refit_minus_truth,
        "fit_sigma": fit_sigma,
        "z_truth": z_truth,
        "within_2sigma": abs(z_truth) <= 2 if np.isfinite(z_truth) else False,
    })

gls_df = pd.DataFrame(gls_rows)
display(gls_df)

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(gls_df))
colors = np.where(gls_df["within_2sigma"], "tab:green", "tab:red")
ax.axhline(0.0, color="black", lw=1.5, label="truth")
ax.axhspan(-2, 2, color="tab:green", alpha=0.12, label="±2 sigma")
ax.scatter(x, gls_df["z_truth"], s=80, c=colors, zorder=3)
ax.set_xticks(x)
ax.set_xticklabels(gls_df["param"], rotation=30, ha="right")
ax.set_ylabel("(GLS refit - truth) / fit_sigma")
ax.set_title(f"Known-red-noise JUG GLS recovery: {CASE_NAME}")
ax.legend(loc="best")
fig.tight_layout()

## Deterministic Difference Check (Secondary)

This is not a likelihood/refit test. It only checks that the truth and sampler PARs differ by the metadata perturbation when evaluated on the same TIM. It is useful for debugging file generation, but the GLS section above is the refit test.

In [ ]:
def residuals_for_params(session, params):
    return session.compute_residuals(params=params, subtract_tzr=False)["residuals_us"]

def perturb_param_dict(params, key, value):
    out = dict(params)
    hp = dict(out.get("_high_precision", {}))
    out[key] = value
    hp.pop(key, None)
    out["_high_precision"] = hp
    return out

# Residual difference using same TIM: stochastic noise cancels.
y_us = np.asarray(truth_res["residuals_us"], dtype=float) - np.asarray(sampler_res["residuals_us"], dtype=float)
r_sampler_us = np.asarray(sampler_res["residuals_us"], dtype=float)
errors_us = np.asarray(sampler_res["errors_us"], dtype=float)
weights = 1.0 / errors_us**2

base_params = dict(sampler_session.params)
base_hp = dict(base_params.get("_high_precision", {}))
base_params["_high_precision"] = base_hp

J_cols = []
fd_steps = []
for pinfo in case["perturbations"]:
    key = pinfo["param"]
    # Step must be large enough to beat TOA string/MJD precision.
    # Using several injected deltas is more stable for tiny spin perturbations.
    h = max(abs(float(pinfo["delta"])) * 5.0, abs(float(pinfo["sigma"])) * 10.0)
    p_plus = perturb_param_dict(base_params, key, float(base_params[key]) + h)
    r_plus_us = np.asarray(residuals_for_params(sampler_session, p_plus), dtype=float)
    J_cols.append((r_plus_us - r_sampler_us) / h)
    fd_steps.append(h)

J = np.column_stack(J_cols)
H = J.T @ (weights[:, None] * J)
b = J.T @ (weights * y_us)
delta_from_residual_difference = np.linalg.solve(H, b)
linear_residual_us = y_us - J @ delta_from_residual_difference

rows_check = []
for i, pinfo in enumerate(case["perturbations"]):
    key = pinfo["param"]
    expected = float(pinfo["true_value"]) - float(pinfo["sampler_value"])
    recovered = float(delta_from_residual_difference[i])
    rows_check.append({
        "param": key,
        "expected_truth_minus_sampler": expected,
        "noise_cancelled_recovered_delta": recovered,
        "difference": recovered - expected,
        "fractional_error": (recovered - expected) / expected if expected != 0 else np.nan,
        "finite_difference_step": fd_steps[i],
    })

injection_check = pd.DataFrame(rows_check)
display(injection_check)
print(f"RMS residual of linear timing-model reconstruction: {np.sqrt(np.mean(linear_residual_us**2)):.6g} us")
print(f"RMS sampler-vs-truth residual difference: {np.sqrt(np.mean(y_us**2)):.6g} us")

## Deterministic Difference Ratio (Secondary)

Target is 1.0. This confirms file-level perturbations, not stochastic refit recovery.

In [ ]:
pass_tolerance = 0.02  # 2 percent fractional error
plot_df = injection_check.copy()
plot_df["recovery_ratio"] = (
    plot_df["noise_cancelled_recovered_delta"] / plot_df["expected_truth_minus_sampler"]
)
plot_df["fractional_error_percent"] = 100.0 * plot_df["fractional_error"]
plot_df["status"] = np.where(
    np.abs(plot_df["fractional_error"]) <= pass_tolerance,
    "PASS",
    "CHECK",
)

display(plot_df[[
    "param",
    "expected_truth_minus_sampler",
    "noise_cancelled_recovered_delta",
    "recovery_ratio",
    "fractional_error_percent",
    "status",
]])

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(plot_df))
colors = np.where(plot_df["status"] == "PASS", "tab:green", "tab:red")
ax.axhline(1.0, color="black", lw=1.5, label="target: exact recovery")
ax.axhspan(1.0 - pass_tolerance, 1.0 + pass_tolerance, color="tab:green", alpha=0.12, label="2% band")
ax.scatter(x, plot_df["recovery_ratio"], s=80, c=colors, zorder=3)
ax.set_xticks(x)
ax.set_xticklabels(plot_df["param"], rotation=30, ha="right")
ax.set_ylabel("recovered correction / expected correction")
ax.set_title(f"Noise-cancelled injection recovery: {CASE_NAME}")
ax.legend(loc="best")
fig.tight_layout()

## Ordinary JUG Refit With Noise Still Present

The next section is useful, but it is **not** a clean injection check. It refits the noisy data from the sampler PAR. A single red-noise realization can project strongly onto spin parameters, especially F0/F1, so the best WLS refit can move away from truth. This is expected behavior unless the fit likelihood includes the same red-noise covariance used to generate the data.

## Run Ordinary JUG Refit

In [ ]:
fit_session = TimingSession(str(sampler_par), str(tim), verbose=False)
fit_result = fit_session.fit_parameters(
    fit_params=fit_params,
    max_iter=MAX_ITER,
    verbose=False,
)

print("converged:", fit_result.get("converged"))
print("iterations:", fit_result.get("iterations"))
print(f"prefit RMS : {fit_result.get('prefit_rms', np.nan):.6g} us")
print(f"postfit RMS: {fit_result.get('final_rms', np.nan):.6g} us")

## Parameter Recovery Table

Definitions:

- `sampler_value`: wrong initial value in `*_sampler.par`
- `truth_value`: injected true value in `*_truth.par`
- `refit_value`: JUG fit result starting from sampler PAR
- `truth_minus_sampler`: injected correction the fit should recover
- `refit_minus_sampler`: correction JUG recovered
- `refit_minus_truth`: remaining error after refit
- `z_truth`: `(refit_value - truth_value) / fit_sigma`

In [ ]:
rows = []
final_params = fit_result["final_params"]
uncertainties = fit_result.get("uncertainties", {})

for p in case["perturbations"]:
    key = p["param"]
    sampler_value = float(p["sampler_value"])
    truth_value = float(p["true_value"])
    refit_value = float(final_params[key])
    fit_sigma = float(uncertainties.get(key, np.nan))
    truth_minus_sampler = truth_value - sampler_value
    refit_minus_sampler = refit_value - sampler_value
    refit_minus_truth = refit_value - truth_value
    z_truth = refit_minus_truth / fit_sigma if np.isfinite(fit_sigma) and fit_sigma > 0 else np.nan
    recovered_fraction = refit_minus_sampler / truth_minus_sampler if truth_minus_sampler != 0 else np.nan
    rows.append({
        "param": key,
        "sampler_value": sampler_value,
        "truth_value": truth_value,
        "refit_value": refit_value,
        "par_sigma_used_for_injection": float(p["sigma"]),
        "injected_sigma_shift": float(p["sigma_shift"]),
        "fit_sigma": fit_sigma,
        "truth_minus_sampler": truth_minus_sampler,
        "refit_minus_sampler": refit_minus_sampler,
        "refit_minus_truth": refit_minus_truth,
        "z_truth": z_truth,
        "recovered_fraction": recovered_fraction,
    })

df = pd.DataFrame(rows)
display(df)

summary = df[["param", "refit_minus_truth", "fit_sigma", "z_truth", "recovered_fraction"]].copy()
display(summary)

## Ordinary Noisy Refit Diagnostic

This plot is **not** the injection pass/fail test. It shows what a plain JUG WLS refit does to one noisy realization. Red noise and DMX covariance can pull F0/F1/DM away from injected truth. Large offsets here mean “noise has timing-model leverage,” not “injection failed.”

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(df))

ax.axhline(1.0, color="black", lw=1.5, label="target if noisy WLS landed on truth")
ordinary_ratio = df["refit_minus_sampler"] / df["truth_minus_sampler"]
ax.scatter(x, ordinary_ratio, s=70, color="tab:orange", label="ordinary noisy JUG refit")
ax.set_xticks(x)
ax.set_xticklabels(df["param"], rotation=30, ha="right")
ax.set_ylabel("ordinary refit correction / expected correction")
ax.set_title(f"Plain noisy WLS refit diagnostic: {CASE_NAME}")
ax.legend()
fig.tight_layout()

## Interpretation

Use the **JUG GLS Refit With Known Injected Red Noise** table and z-score plot as the main result. That section asks the right question: if JUG is told the red-noise covariance used to generate the data, does the timing fit recover truth within fit uncertainties?

Use the deterministic difference sections only to debug whether the files encode the requested perturbation.

Caveat for DM: these NG15 files include DMX ranges, and JUG auto-fits DMX bins. Global DM is therefore not independently identifiable in the same way as F0/F1 unless DMX handling is controlled. A large DM z-score in the GLS section is a degeneracy warning, not necessarily a noise-injection failure.

In [ ]:
# Optional: inspect high-z parameters quickly
threshold = 2.0
bad = df[np.abs(df["z_truth"]) > threshold]
if len(bad):
    print(f"Parameters with |z_truth| > {threshold}:")
    display(bad[["param", "refit_minus_truth", "fit_sigma", "z_truth", "recovered_fraction"]])
else:
    print(f"All injected parameters have |z_truth| <= {threshold}.")